# C-05 Data Distribution Analysis

**Control:** C-05 Data Distribution Analysis, MAK-AI Responsible AI Toolkit.

**Learning goal:** see who is over- or under-represented in a dataset, including in intersections of groups; find groups too small to learn from; and decide whether to collect more data or sample differently.

**Time to run:** about 1 minute on an ordinary laptop. No GPU.

**Licence:** this notebook is Apache-2.0.

**Example data:** Statlog (German Credit Data) (Hofmann, 1994), DOI 10.24432/C5NC77, licence CC BY 4.0, 1,000 loan applications, from https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data. It is credited here because CC BY 4.0 asks for attribution.

**A caveat before you start:** 1,000 applications from one German bank, released in 1994. The file combines sex and marital status in one code, which is a lesson in measurement: the notebook reads sex from that code and loses marital status for women. The reference shares in the "Adapt here" cell are **illustrative only**, invented for the example. Use shares for the people your system will serve.

## Linked risk areas and questions

Generated from content 5.2.0. Do not edit by hand.

Risk areas: 3

- PP-02: Data Scarcity & Availability Bias
- PP-04: Population Bias
- PP-05: Sampling & Generalization Bias

Questions: 10: Q-PP-03, Q-PP-04, Q-PP-05, Q-PP-24, Q-PP-09, Q-PP-16, Q-PP-17, Q-PP-18, Q-PP-20, Q-PP-22

## When to use this notebook

Use it when you are answering questions about who is in your data and your status is **Not yet** or **Partly**, especially with the reason code `ny_needs_skills` ("Needs skills or guidance the team does not have yet"). It helps with:

- `Q-PP-09`: is your training data representative of the target population?
- `Q-PP-18`: could your data collection methods introduce selection bias?
- `Q-PP-22`: does your data represent edge cases and rare conditions?
- `Q-PP-16`: the options on class imbalance analysis and distribution analysis across subgroups.

You need a table with one row per person and the columns that define the groups you worry about.

In [ ]:
import hashlib
import importlib.metadata
import io
import os
import platform
import tempfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chisquare

SEED = 20261030  # the one random seed; this notebook has no random step, but the seed is kept for the record
np.random.seed(SEED)
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 80)

PACKAGES = ['pandas', 'numpy', 'scipy', 'matplotlib']

print('Python', platform.python_version())
for _package in PACKAGES:
    print(_package, importlib.metadata.version(_package))

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/144/data.csv'
DATA_SHA256 = 'f985b42636c9e28ce4fb3913738d1ebfd0f7947fda8d4406fe2ded5d84295acb'
DATA_FILE = 'uci-german-credit-144-data.csv'  # the cache file name is fixed here

CACHE_DIR = Path.home() / '.cache' / 'makrai-notebooks'


MAX_DOWNLOAD_BYTES = 64 * 1024 * 1024  # a dataset file is never larger than this; a longer body is refused


def fetch_checked(url, sha256, file_name):
    """Return the bytes of a dataset whose SHA-256 matches `sha256`.

    The file is downloaded once into the cache folder, and written there only after its hash
    matches. Every run, including the first, hashes the bytes before they are used, and a
    mismatch stops the notebook. The body is read in 1 MiB pieces and refused past MAX_DOWNLOAD_BYTES.
    `file_name` is fixed in this notebook, never taken from the address.
    """
    if not url.startswith('https://'):
        raise ValueError(f'the dataset address must be https: {url}')
    if Path(file_name).name != file_name:
        raise ValueError(f'the cache file name must be a plain name: {file_name}')

    def read_checked(stream):
        digest, pieces, size = hashlib.sha256(), [], 0
        for piece in iter(lambda: stream.read(1 << 20), b''):
            size += len(piece)
            if size > MAX_DOWNLOAD_BYTES:
                raise RuntimeError(f'{file_name}: the file is larger than {MAX_DOWNLOAD_BYTES:,} bytes; refused')
            digest.update(piece)
            pieces.append(piece)
        return b''.join(pieces), digest.hexdigest()

    path = CACHE_DIR / file_name
    if path.exists():
        with path.open('rb') as stream:
            data, found = read_checked(stream)
        source, remedy = f'the cached file {path}', f'Delete {path} and run again.'
    else:
        request = urllib.request.Request(url, headers={'User-Agent': 'makrai-notebooks'})
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError(f'{url} redirected to a non-https address: {response.geturl()}')
                data, found = read_checked(response)
        except OSError as error:
            raise RuntimeError(f'could not download {url}: {error}. If you cannot reach it, save the file by hand '
                               f'as {path}; it is checked against the pinned SHA-256 either way.') from error
        source, remedy = url, 'Do not use this download; check the dataset page for a new version.'
    if found != sha256:
        raise RuntimeError(f'{file_name}: the SHA-256 of {source} is {found}, expected {sha256}. {remedy}')
    if not path.exists():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        partial = tempfile.NamedTemporaryFile(dir=CACHE_DIR, suffix='.part', delete=False)
        try:
            with partial:
                partial.write(data)
            os.replace(partial.name, path)  # the cache never holds a half-written file, whoever else is writing
        except BaseException:
            Path(partial.name).unlink(missing_ok=True)
            raise
    return data


# The file has no column names and codes such as A11. Names and meanings are from the dataset's own description.
COLUMNS = ['status_checking', 'duration_months', 'credit_history', 'purpose', 'credit_amount', 'savings',
           'employment_since', 'instalment_rate', 'personal_status_sex', 'other_debtors', 'residence_since',
           'property', 'age', 'other_instalment_plans', 'housing', 'existing_credits', 'job', 'people_liable',
           'telephone', 'foreign_worker']
MEANING = {
    'A11': 'under 0 DM', 'A12': '0 to 200 DM', 'A13': '200 DM or more', 'A14': 'no checking account',
    'A30': 'no credits or all paid back', 'A31': 'all paid back at this bank', 'A32': 'paid back duly so far',
    'A33': 'delay in the past', 'A34': 'critical account or other credits',
    'A40': 'car (new)', 'A41': 'car (used)', 'A42': 'furniture or equipment', 'A43': 'radio or television',
    'A44': 'domestic appliances', 'A45': 'repairs', 'A46': 'education', 'A48': 'retraining', 'A49': 'business',
    'A410': 'other purpose',
    'A61': 'under 100 DM', 'A62': '100 to 500 DM', 'A63': '500 to 1000 DM', 'A64': '1000 DM or more', 'A65': 'unknown or none',
    'A71': 'unemployed', 'A72': 'under 1 year', 'A73': '1 to 4 years', 'A74': '4 to 7 years', 'A75': '7 years or more',
    'A91': 'male, divorced or separated', 'A92': 'female, divorced, separated or married', 'A93': 'male, single',
    'A94': 'male, married or widowed', 'A95': 'female, single',
    'A101': 'none', 'A102': 'co-applicant', 'A103': 'guarantor',
    'A121': 'real estate', 'A122': 'savings agreement or life insurance', 'A123': 'car or other', 'A124': 'unknown or none',
    'A141': 'bank', 'A142': 'stores', 'A143': 'none',
    'A151': 'rent', 'A152': 'own', 'A153': 'for free',
    'A171': 'unemployed or unskilled, non-resident', 'A172': 'unskilled, resident', 'A173': 'skilled employee',
    'A174': 'management or highly qualified',
    'A191': 'none', 'A192': 'yes', 'A201': 'yes', 'A202': 'no',
}

# To use your own file, replace the next lines with:  raw = Path('your-file.csv').read_bytes()  and  df = pd.read_csv(io.BytesIO(raw))
raw = fetch_checked(DATA_URL, DATA_SHA256, DATA_FILE)
DATA_SHA256_USED = hashlib.sha256(raw).hexdigest()  # the evidence summary records the hash of the bytes used
df = pd.read_csv(io.BytesIO(raw))
df.columns = COLUMNS + ['class']  # German Credit only (delete the decoding and the two derived columns below for your own data)
for _column in df.columns[:-1]:
    if df[_column].dtype == object or str(df[_column].dtype).startswith('str'):
        df[_column] = df[_column].map(MEANING)
# Two columns made for this notebook. The file gives sex and marital status as ONE code, so sex is read from it, and
# marital status is lost for women. That is the measurement lesson the "not shown" cell comes back to.
df['sex'] = np.where(df['personal_status_sex'].str.startswith('female'), 'female', 'male')
df['age_band'] = pd.cut(df['age'], [0, 24, 34, 44, 59, 200], labels=['under 25', '25 to 34', '35 to 44', '45 to 59', '60 and over']).astype(str)
print(f'{len(df):,} rows, {df.shape[1]} columns; SHA-256 of the data used: {DATA_SHA256_USED[:12]}...')

In [ ]:
# ADAPT HERE. Edit only this cell to run the notebook on your own data. Every later cell reads only these names.
LABEL = 'class'                      # the column that holds the outcome the model will learn
SMALLER_CLASS = 2                    # the value of LABEL that is rarer or costlier to miss (here 2 is a bad credit risk)
GROUP_COLUMNS = ['sex', 'foreign_worker', 'age_band']   # the columns that define the groups you look at
INTERSECTION = ['age_band', 'sex', 'foreign_worker']    # columns to cross, in this order
MIN_GROUP_SIZE = 30                  # groups smaller than this are flagged, not interpreted; 30 is the example's value, not advice: choose yours in the stop-and-decide cell
MIN_CLASS_EXAMPLES = 20              # groups with fewer examples of either outcome are flagged; 20 is the example's value, not advice
# Illustrative shares, invented for this example. Replace them with shares for the people your system will serve.
REFERENCE_SHARES = {
    'sex': {'female': 0.50, 'male': 0.50},
    'age_band': {'under 25': 0.25, '25 to 34': 0.25, '35 to 44': 0.20, '45 to 59': 0.20, '60 and over': 0.10},
}

### Step 1: how many people are in each group

**What this does.** For each column in `GROUP_COLUMNS` it counts the people in each group and shows the share. Groups with fewer than `MIN_GROUP_SIZE` people are flagged.

**What it shows.** Who is common and who is rare in the data.

**What it does not show.** Whether rare groups are rare in the world or only in this dataset, and whether a group is missing altogether because nobody recorded it.

In [ ]:
count_rows = []
for column in GROUP_COLUMNS:
    counts = df[column].fillna('missing').astype(str).value_counts()
    for group, number in counts.items():
        count_rows.append({'column': column, 'group': group, 'people': int(number), 'share': number / len(df), 'small': number < MIN_GROUP_SIZE})
group_counts = pd.DataFrame(count_rows)
print(group_counts.round(3).to_string(index=False))

### Step 2: intersections, where groups get small

**What this does.** It crosses the columns in `INTERSECTION` (here age band by sex by foreign-worker status) with `pd.crosstab`, so each cell is one combination, and flags cells with fewer than `MIN_GROUP_SIZE` people. It reports how many cells are small or empty, and what share of people live in a small cell.

**What it shows.** That groups which look large one column at a time can be tiny together. A model that seems to work for "women" and for "foreign workers" can have almost no examples of foreign women over 60. The number of cells grows fast with every column you add.

**What it does not show.** Which small cells matter. A small cell may be a group your use cares about a great deal. It also does not show combinations of the row columns that nobody in the data has: such a combination is not a row of the table, so it is not counted among the cells or the empty ones. Check the row labels for combinations you expected to see.

In [ ]:
cross_columns = [df[c].fillna('missing').astype(str) for c in INTERSECTION]
crossed = pd.crosstab(cross_columns[:-1], cross_columns[-1])
cells = crossed.stack()
small_cells = cells[cells < MIN_GROUP_SIZE]
share_in_small = small_cells.sum() / cells.sum()
print(crossed.to_string())
print()
print(f'cells: {len(cells)}; smaller than {MIN_GROUP_SIZE}: {len(small_cells)}; empty: {int((cells == 0).sum())}')
print(f'share of people who sit in a small cell: {share_in_small:.1%}')

### Step 3: class balance by group

**What this does.** For each group it shows the share of people with the outcome you named in `SMALLER_CLASS`, and how many examples of each outcome the group has. A group is flagged when either outcome has fewer than `MIN_CLASS_EXAMPLES` examples.

**What it shows.** Whether the outcome is rarer in some groups, and whether a group has too few examples of one outcome for a model to learn from. A model can look accurate on a group by always predicting the common outcome.

**What it does not show.** Why the rates differ. A different rate may be a real difference or a difference in how the outcome was recorded (see C-04).

In [ ]:
balance_rows = []
for column in GROUP_COLUMNS:
    labels = df[column].fillna('missing').astype(str)
    for group, part in df.groupby(labels):
        smaller = int((part[LABEL] == SMALLER_CLASS).sum())
        larger = len(part) - smaller
        balance_rows.append({'column': column, 'group': group, 'people': len(part), 'share with the smaller class': smaller / len(part),
                             'examples of the smaller class': smaller, 'examples of the other class': larger,
                             'too few examples of a class': min(smaller, larger) < MIN_CLASS_EXAMPLES})
class_balance = pd.DataFrame(balance_rows)
print(f'overall share with the smaller class: {(df[LABEL] == SMALLER_CLASS).mean():.3f}')
print(class_balance.round(3).to_string(index=False))

### Step 4: comparison with reference shares you supply

**What this does.** For each column named in `REFERENCE_SHARES` it compares the data's shares with the shares you gave: the difference in percentage points, the ratio, and a chi-square goodness-of-fit test from SciPy (`chisquare`) of whether the data's counts could have come from the reference shares. The example shares are **illustrative**: they stand for what a team would take from a census, a registry or a service's records. Only the groups you name in `REFERENCE_SHARES` are compared: people in any other group, including missing values, are left out of this step, and the data shares are shares of the people who remain.

**What it shows.** Which groups the data over- or under-represents compared with the reference.

**What it does not show.** Whether the reference is right. A national census is the wrong reference for a service that reaches one district (see C-03). With small samples the test cannot reject much, and with large samples it rejects everything, so read the differences first.

In [ ]:
reference_rows, fit_rows = [], []
for column, shares in REFERENCE_SHARES.items():
    labels = df[column].fillna('missing').astype(str)
    observed = labels.value_counts().reindex(list(shares), fill_value=0)
    expected = pd.Series(shares) / sum(shares.values()) * observed.sum()
    statistic, p_value = chisquare(f_obs=observed, f_exp=expected)
    fit_rows.append({'column': column, 'chi-square': statistic, 'degrees of freedom': len(shares) - 1, 'p-value': p_value})
    for group in shares:
        reference_rows.append({'column': column, 'group': group, 'data share': observed[group] / observed.sum(),
                               'reference share (illustrative)': expected[group] / observed.sum(),
                               'difference (points)': 100 * (observed[group] - expected[group]) / observed.sum(),
                               'data share / reference share': observed[group] / expected[group]})
reference_comparison = pd.DataFrame(reference_rows)
goodness_of_fit = pd.DataFrame(fit_rows)
print(reference_comparison.round(3).to_string(index=False))
print()
print(goodness_of_fit.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
shown = group_counts.assign(label=group_counts['column'] + ': ' + group_counts['group']).sort_values('people')
axes[0].barh(shown['label'], shown['people'], color=np.where(shown['small'], 'tab:red', 'tab:blue'))
axes[0].axvline(MIN_GROUP_SIZE, linestyle='--', color='grey')
axes[0].set_title('People per group (red: under the minimum)')
axes[1].imshow(crossed.to_numpy() < MIN_GROUP_SIZE, cmap='Reds', aspect='auto')
axes[1].set_yticks(range(len(crossed)))
axes[1].set_yticklabels([' / '.join(map(str, i)) for i in crossed.index], fontsize=7)
axes[1].set_xticks(range(crossed.shape[1]))
axes[1].set_xticklabels(crossed.columns, fontsize=8)
axes[1].set_title('Small cells of the intersection (dark: under the minimum)')
balance = class_balance.assign(label=class_balance['column'] + ': ' + class_balance['group']).sort_values('share with the smaller class')
axes[2].barh(balance['label'], balance['share with the smaller class'])
axes[2].set_title('Share with the smaller class')
fig.tight_layout()

### Stop and decide: the smallest group you will accept, and what you will do about smaller ones

The notebook flags small groups against a size you set. It cannot tell you what size is enough.

Questions to ask in the room:

1. How many people does a group need before you would trust a number about it? You set `MIN_GROUP_SIZE` before looking: was it chosen for the statistics, or did the table choose it for you?
2. For each group below that size: can you collect more, combine it with another group (and say what that hides), sample differently, or state that the model must not be used on that group?
3. Which small groups are small in the world (a rare condition) and which are small only in your data (nobody asked)? Who can tell you?
4. Which groups matter to your use but are not a column at all?
5. If the outcome is rare in a group, how will you know the model works for it, and not just says "no" every time?

Write your choices in the next cell. Leave it blank if you have not decided, and the summary will say so.

In [ ]:
# YOUR CHOICES. The notebook does not fill this in. Plain words are fine.
SMALLEST_GROUP_WE_ACCEPT = ''      # the number, and how you chose it
WHAT_WE_DO_ABOUT_SMALLER_GROUPS = ''   # collect more, combine, sample differently, or do not use the model on them
GROUPS_THAT_MATTER_BUT_ARE_NOT_COLUMNS = ''   # groups you cannot check with this data
REFERENCE_WE_USED = ''             # where the reference shares came from, or that they were illustrative

In [ ]:
import zipfile

EVIDENCE_DIR = Path('evidence')
EVIDENCE_DIR.mkdir(exist_ok=True)

# Small tables of counts and shares. They hold no row of the data.
group_counts.round(4).to_csv(EVIDENCE_DIR / 'C-05_group_counts.csv', index=False)
crossed.to_csv(EVIDENCE_DIR / 'C-05_intersection.csv')
class_balance.round(4).to_csv(EVIDENCE_DIR / 'C-05_class_balance.csv', index=False)
reference_comparison.round(4).to_csv(EVIDENCE_DIR / 'C-05_reference_comparison.csv', index=False)
goodness_of_fit.round(6).to_csv(EVIDENCE_DIR / 'C-05_goodness_of_fit.csv', index=False)
fig.savefig(EVIDENCE_DIR / 'C-05_distribution.png', dpi=100, bbox_inches='tight')

versions = '\n'.join(f'- {package} {importlib.metadata.version(package)}' for package in PACKAGES)
summary = f"""# C-05 Data Distribution Analysis: evidence summary

Attach the file `C-05_evidence.zip`, which the notebook writes next to the `evidence` folder, to the decision it supports (questions Q-PP-09, Q-PP-16, Q-PP-18, Q-PP-22). It holds this notebook's own summary, tables and pictures, counts and shares only, never the data.

## What was run
- Rows: {len(df):,}; groups: {', '.join(GROUP_COLUMNS)}; intersection: {' by '.join(INTERSECTION)}
- Flagging sizes used (set in the adapt-here cell; not a decision): groups under {MIN_GROUP_SIZE} people; fewer than {MIN_CLASS_EXAMPLES} examples of a class
- Groups under the minimum: {int(group_counts['small'].sum())} of {len(group_counts)}
- Intersection cells under the minimum: {len(small_cells)} of {len(cells)}, holding {share_in_small:.1%} of people
- Groups with too few examples of a class: {int(class_balance['too few examples of a class'].sum())} of {len(class_balance)}
- Reference shares: illustrative unless the cell below says otherwise

## Choices made in the stop-and-decide cell
- Smallest group we accept: {SMALLEST_GROUP_WE_ACCEPT or 'not written yet'}
- What we do about smaller groups: {WHAT_WE_DO_ABOUT_SMALLER_GROUPS or 'not written yet'}
- Groups that matter but are not columns: {GROUPS_THAT_MATTER_BUT_ARE_NOT_COLUMNS or 'not written yet'}
- Reference used: {REFERENCE_WE_USED or 'not written yet'}

## Files beside this summary
C-05_group_counts.csv, C-05_intersection.csv, C-05_class_balance.csv, C-05_reference_comparison.csv, C-05_goodness_of_fit.csv, C-05_distribution.png

## Package versions
- Python {platform.python_version()}
{versions}

## Data
SHA-256 of the data file used: {DATA_SHA256_USED}
"""
(EVIDENCE_DIR / 'C-05_summary.md').write_text(summary, encoding='utf-8')
# One ZIP of the files this cell wrote, next to the folder, not inside it. Only the files this cell wrote enter it.
# It is built under a .part name and moved into place only when complete, so an error while it is built leaves no ZIP.
Path('C-05_evidence.zip').unlink(missing_ok=True)
Path('C-05_evidence.zip.part').unlink(missing_ok=True)
with zipfile.ZipFile('C-05_evidence.zip.part', 'x', zipfile.ZIP_DEFLATED) as archive:
    for name in [
        'C-05_class_balance.csv', 'C-05_distribution.png', 'C-05_goodness_of_fit.csv',
        'C-05_group_counts.csv', 'C-05_intersection.csv', 'C-05_reference_comparison.csv',
        'C-05_summary.md'
    ]:
        path = EVIDENCE_DIR / name
        if not path.is_file() or path.is_symlink():
            raise RuntimeError(f'{name} is missing or is a link, so no ZIP was made. Run this notebook again from the top; if the file is a link, delete it first.')
        archive.write(path, name)
os.replace('C-05_evidence.zip.part', 'C-05_evidence.zip')
print(summary)

## What this does not show, and where to go next

- **A count is not representation.** Enough people in a group does not mean they are the right people (see C-03 for who is missing, and C-04 for how the label was made).
- **Groups are choices.** The file gives sex and marital status as one code, so this notebook reads sex from it and cannot say anything about marital status for women. A different coding would have allowed different questions.
- **The reference shares here are invented.** They show how to compare. They say nothing about any real population.
- **More data is not always the answer.** If a group is small because it is small in the world, you may need a different design, a human check, or a decision not to use the model on that group.
- **Intersections multiply.** Each column you add multiplies the cells. Choose the combinations that matter for your use before looking at which ones are large.

Where to go next: C-03 Data Quality & Representativeness Audit (completeness and the right reference), C-04 Bias Detection in Data (label rates and proxies), C-14 Disaggregated Performance Analysis (what small groups do to performance numbers), and the datasheet template, which records how the data was collected and who is in it.